# A2.2 — Huấn luyện U-Net (ResNet-50) và SegFormer-B2 trên Google Colab

Notebook chạy các thí nghiệm A2.2: 2 mô hình × 4 lượng dữ liệu train (100%, 50%, 25%, 10%), rồi đánh giá thêm hai mô hình 100% (đổi độ phân giải test, ảnh bị nhiễu/mờ/nén JPEG).

**Cách dùng**
1. **Runtime → Change runtime type → T4 GPU**.
2. Ở ô cấu hình, chọn `ARCH`: hai thành viên có thể chạy song song trên hai tài khoản, mỗi người một mô hình (`"unet_r50"` và `"segformer_b2"`). Để `None` thì chạy cả hai.
3. **Runtime → Run all**. Log và checkpoint lưu vào Google Drive (`MyDrive/CO5085_A2`). Nếu Colab bị ngắt, mở lại và **Run all**: lần chạy đã xong được bỏ qua, lần đang dở chạy tiếp từ epoch cuối.
4. Ô cuối tải về file zip gồm log JSON và trọng số của mô hình 100% để phân tích trên máy (`analyze.py`).

In [ ]:
# Configuration: which architecture this Colab account trains (None = both)
ARCH = None            # "unet_r50", "segformer_b2" or None
EPOCHS = None          # None = config.TRAIN_HPARAMS["epochs"]; set a smaller number only if GPU time runs out

In [ ]:
# Check that a GPU is attached (should show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Mount Google Drive: logs, checkpoints and the dataset archives are kept there so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/CO5085_A2/runs"
RAW_DIR = "/content/drive/MyDrive/CO5085_A2/raw"
!mkdir -p "$RUN_DIR" "$RAW_DIR"

In [ ]:
# Get the latest code from GitHub
REPO = "/content/Deep_Learning_Computer_Vision_CO5085"
!if [ -d "$REPO" ]; then git -C "$REPO" pull -q; else git clone -q https://github.com/dqpinfo/Deep_Learning_Computer_Vision_CO5085.git "$REPO"; fi
%cd {REPO}/Assignment_2
!git log --oneline -1

# The two archives (~800 MB) are cached on Drive; the images are extracted to the fast local disk of the session
!mkdir -p data && ln -sfn "$RAW_DIR" data/raw

In [ ]:
# Colab already has torch, torchvision and transformers; only timm is missing
!pip install -q timm
# Download (first time only, checked with MD5), extract and load the fixed split
!python -c "from data import get_split; tr, va, te = get_split(); print(len(tr), len(va), len(te))"

In [ ]:
# Quick pipeline check on 64 images, 1 epoch (about 1 minute); not used for reported results
SMOKE_EXP = f"{ARCH or 'segformer_b2'}_f100"
!python train.py --exp {SMOKE_EXP} --subset 64 --epochs 1 --workers 2

In [ ]:
# Run the experiments. Rerun this cell after a disconnect: finished runs are skipped, the interrupted one resumes.
args = f"--run-dir '{RUN_DIR}' --workers 2"
if ARCH:
    args += f" --arch {ARCH}"
if EPOCHS:
    args += f" --epochs {EPOCHS}"
!python run_all.py {args}

In [ ]:
# Extra evaluations of the 100% models: test resolution and corruptions (about 10-15 minutes per model)
import os
for arch in ([ARCH] if ARCH else ["unet_r50", "segformer_b2"]):
    if os.path.exists(f"{RUN_DIR}/{arch}_f100_best.pth") and not os.path.exists(f"{RUN_DIR}/extra_{arch}.json"):
        !python eval_extra.py --arch {arch} --run-dir "{RUN_DIR}" --workers 2

In [ ]:
# Summary of the finished runs
import json, glob
print(f"{'Run':<20}{'test mIoU':>10}{'pet IoU':>9}{'giống/ảnh':>10}{'best ep':>8}{'train (min)':>12}{'mem (GB)':>9}{'img/s':>7}")
for path in sorted(glob.glob(f"{RUN_DIR}/*_f*.json")):
    if path.endswith("_test_images.json"):
        continue
    log = json.load(open(path))
    t = log["test"]
    print(f"{log['experiment']['name']:<20}{t['miou']:>10.4f}{t['pet_iou']:>9.4f}{t['breed_acc_image']:>10.4f}{log['best_epoch']:>8}"
          f"{log['train_time_sec'] / 60:>12.1f}{log.get('peak_gpu_mem_mb', 0) / 1024:>9.1f}{log['inference_img_per_sec']:>7.0f}")

In [ ]:
# Download the logs (+ the weights of the 100% models, ~110-130 MB each) for analyze.py and the demo
!cd "$RUN_DIR" && zip -q -j /content/A2_runs.zip *.json *_f100_best.pth
from google.colab import files
files.download("/content/A2_runs.zip")